# UQTS-2026: Alpha Discovery Lab
## Phase 1: Signal vs. Fluid Framework

This notebook demonstrates the foundational components of the **UQTS-2026** research lab:
1. **Bi-temporal Data Engine**: Point-in-Time (PIT) consistency.
2. **Fractional Differentiation**: Stationarity with memory preservation.
3. **Wavelet Spectrograms**: Multi-resolution spectral analysis.
4. **Residualized Alpha**: Idiosyncratic target labeling (Multi-Ticker).
5. **Alpha Ranker**: RankNet LTR model for cross-sectional scoring.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datetime import datetime, timedelta

from research_lab.data_engine import DataEngine
from research_lab.alpha_core import FractionalDifferencer, WaveletFeatureGenerator
from research_lab.alpha_labeler import AlphaLabeler
from research_lab.alpha_ranker import RankNet

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = [12, 8]

### 1. Data Ingestion (Bi-temporal PIT)
We generate synthetic data with a 'Knowledge Time' delay to verify zero look-ahead bias.

In [ ]:
engine = DataEngine()
tickers = ['AAPL', 'MSFT', 'GOOG', 'SPY']
engine.generate_synthetic_pit_data(tickers, days=500)

# Fetch a PIT view as of midway through the period
as_of_date = datetime(2020, 1, 1) + timedelta(days=250)
pit_view = pd.concat([engine.get_pit_view(t, as_of_date) for t in tickers])

print(f"Data points known for universe as of {as_of_date}: {len(pit_view)}")
pit_view.head()

### 2. Fractional Differentiation ($d=0.4$)
Stationarizing the series while preserving long-term memory.

In [ ]:
fd = FractionalDifferencer(d=0.4)
aapl_view = engine.get_pit_view('AAPL', as_of_date)
aapl_stationary = fd.transform(aapl_view['close'])

fig, ax1 = plt.subplots()

ax1.set_xlabel('Event Time')
ax1.set_ylabel('Original Price', color='tab:blue')
ax1.plot(aapl_view['event_time'], aapl_view['close'], color='tab:blue', label='Original')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.set_ylabel('Stationary (d=0.4)', color='tab:red')
ax2.plot(aapl_view['event_time'], aapl_stationary, color='tab:red', label='Stationary')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('AAPL: Original vs. Fractionally Differenced')
fig.tight_layout()
plt.show()

### 3. Wavelet Spectrogram (Morlet)
Analyzing the signal across dyadic scales to detect regimes.

In [ ]:
wfg = WaveletFeatureGenerator()
spectrogram = wfg.generate(aapl_stationary)

plt.imshow(spectrogram, aspect='auto', cmap='jet', extent=[0, len(aapl_stationary), 1, 8])
plt.colorbar(label='Amplitude')
plt.title('Market Spectrogram: AAPL (Dyadic Scales 2^1 to 2^8)')
plt.ylabel('Log Scale')
plt.xlabel('Time Steps')
plt.show()

### 4. Multi-Ticker Residualized Alpha & Z-Scoring
Full pipeline: Calculate Returns -> Residualize vs SPY -> Cross-sectional Z-Score.

In [ ]:
labeler = AlphaLabeler()

# 1. Generate forward returns (horizon=5 for demo)
returns_df = labeler.generate_labels(pit_view, horizon=5)

# 2. Residualize Universe against SPY
market_proxy = returns_df['SPY']
asset_returns = returns_df.drop(columns=['SPY'])
residuals = labeler.residualize_universe(asset_returns, market_proxy)

# 3. Apply Cross-sectional Z-Score
final_labels = labeler.apply_z_score(residuals)

print("Final Multi-Ticker RankNet Target Labels (Z-Scores):")
display(final_labels.tail())

# Verify zero mean per row (cross-section)
row_means = final_labels.mean(axis=1)
print(f"\nAverage Cross-sectional Mean: {row_means.mean():.4e} (Should be ~0)")

# Plot Target Distribution
final_labels.plot(kind='hist', bins=50, alpha=0.5)
plt.title('Final Label Distribution (Z-Scores)')
plt.show()

### 5. Alpha Ranker (RankNet LTR)
Inference with the RankNet model.

In [ ]:
# Input features: 8 wavelet scales
model = RankNet(input_dim=8)
model.eval()

# Sample input (spectrogram at last timestep)
sample_input = torch.tensor(spectrogram[:, -1]).unsqueeze(0).float()
with torch.no_grad():
    score = model(sample_input)

print(f"RankNet Output Score for AAPL (Latest): {score.item():.4f}")